# ODSC 2026 — Notebook 1: Custom Claw

Compliance is a real issue for virtually every company. Entire departmentsExists for the sole purpose of ensuring that compliance is up to date for every vertical in every region they do business. Here we will build a compliance monitor from scratch. We will focus on licensure compliance for graduating educators from across the United States Wishing to teach in private or public institutions. We will build an agent using the Principle of Least Privilege (PoLP). ~150 lines of Python that fetches state DOE pages, hashes the content, detects changes, sends Telegram alerts and updates our compliance database.

| Part | What you build |
| --- | --- |
| **Part 1** | Environment setup |
| **Part 2** | Custom Claw — fetch, hash, diff, alert |
| **Part 3** | Extend with Brave Search *(optional)* |
| **Part 3b** | Minimum viable conversational agent *(optional)* |
| **Part 4** | Telegram notifications |

**Up next:** `openclaw.ipynb` — the same logic in a production agent framework.

---
## Part 1: Environment Setup

In [1]:
import subprocess
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
BRAVE_API_KEY  = os.getenv("BRAVE_API_KEY", "")

assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"

if not BRAVE_API_KEY or BRAVE_API_KEY.startswith("your-"):
    print("⚠️  BRAVE_API_KEY not set — Parts 3 and 3b will be skipped")
    BRAVE_API_KEY = ""
else:
    print("✅ Brave key set")

docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

print("✅ OpenAI key set")
print("✅ Docker running")
print("\nReady to go.")

✅ Brave key set
✅ OpenAI key set
✅ Docker running

Ready to go.


---
## Part 2: From LLM to Agent

A cron job fetches a page and fires an alert. An **agent** fetches a page, understands what changed, and explains why it matters for teachers.

The difference is one layer: give the LLM tools it can call. We'll build this up step by step so you can see exactly where the magic happens.

### Step 1: The Base LLM

Just an OpenAI call. It knows a lot from training — but it can't browse live pages or remember anything between runs.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What Praxis tests are required for Math 7-12 in Ohio?"}]
)
print(response.choices[0].message.content)
# Gives training-data knowledge — can't verify today's Ohio DOE page.

### Step 2: Add a Skill

Decorate any Python function with `agent.tool_plain()` and the LLM can call it. **One line turns a function into a skill.**

In [ ]:
import sys; sys.path.insert(0, "custom-claw/src")

from pydantic_ai import Agent
from skills.fetch_page import fetch_compliance_page

agent = Agent(
    model="openai:gpt-4o-mini",
    instructions="You monitor teacher licensure compliance pages."
)

agent.tool_plain(fetch_compliance_page)  # ← one line = one new skill

result = await agent.run(
    "Fetch https://education.ohio.gov/Topics/Teaching/Licensure "
    "and summarize what it says about Praxis requirements."
)
print(result.output)

### Step 3: Add More Skills

Each new capability is one function. The agent decides when and how to call each one — you don't write the routing logic.

In [ ]:
from skills.check_change import check_for_change
from skills.send_alert import send_alert
from skills.discover_urls import discover_urls

# Open custom-claw/src/skills/ — each file is one skill.
# Adding a new one: write a function, register it below.

agent.tool_plain(check_for_change)  # compare page content to stored snapshot
agent.tool_plain(send_alert)        # fire a Telegram alert
agent.tool_plain(discover_urls)     # Brave Search for new compliance pages

### Step 4: Natural Language In, Compliance Check Out

In [ ]:
from agent import run  # imports the fully-configured agent

result = await run("Check Ohio and Texas for any teacher licensure changes and alert me if anything changed.")
print(result)

### The Ceiling

To get here we wrote:
- SSRF-safe URL fetching with IP pinning
- SHA-256 content hashing + SQLite snapshot diffing  
- Telegram alerting with proper HTML escaping
- A tool-routing loop the agent orchestrates

**Adding a new skill is one file + one line. That's genuinely easy.**

What isn't easy: owning all the plumbing above forever. Networking, storage, error handling, auth, scheduling — you wrote it, you maintain it.

> **Up next:** OpenClaw does all of this from a single chat message. No plumbing required.

---
## Part 3: Extending Custom Claw with Brave Search *(Optional)*

**Skip this part if you don't have a Brave API key** — free at [brave.com/search/api](https://brave.com/search/api/) but not required for the rest of the workshop.

### The limitation we're solving

Custom Claw's watch list is hardcoded in `compliance_urls.py`. If a state adds a new page, or we want to monitor a district we've never seen before, we have to edit the code.

What if the agent could *find* the right URLs itself?

### The idea

Add a `discover_urls(state)` function that:
1. Queries Brave Search for `"{state} teacher licensure certification requirements site:.gov"`
2. Returns the top results as candidate URLs
3. Feeds them into the existing hash/diff/alert flow

This is exactly what OpenClaw's `brave-search` skill does automatically when you ask it to monitor a new state — no code changes needed. We're building a manual version of that here to show the concept.

In [ ]:
import requests

BRAVE_API_KEY = os.getenv("BRAVE_API_KEY", "")

def discover_compliance_urls(state: str, max_results: int = 3) -> list[dict]:
    if not BRAVE_API_KEY:
        print("⚠️  BRAVE_API_KEY not set — skipping URL discovery")
        return []

    query = f"{state} teacher licensure certification requirements site:.gov"
    resp = requests.get(
        "https://api.search.brave.com/res/v1/web/search",
        headers={"Accept": "application/json", "X-Subscription-Token": BRAVE_API_KEY},
        params={"q": query, "count": max_results},
        timeout=10,
    )
    resp.raise_for_status()

    results = resp.json().get("web", {}).get("results", [])
    return [{"title": r["title"], "url": r["url"]} for r in results]


# --- Demo ---
state = "Indiana"
print(f"Discovering compliance URLs for {state}...\n")
discovered = discover_compliance_urls(state)

if discovered:
    for r in discovered:
        print(f"  {r['title']}")
        print(f"  {r['url']}\n")
    print("These URLs could be passed directly into Custom Claw's run_check() —")
    print("no hardcoded watch list needed.")
else:
    print("(Brave API not configured — in the full setup, results would appear here.)\n")
    print("With OpenClaw, you'd just say:")
    print('  \"Check for compliance updates in Indiana\"')
    print("...and it searches, finds the pages, monitors them, and alerts you automatically.")

---
### Part 3b: Minimum Viable Conversational Agent *(Optional)*

We added Brave Search, but it's still structured code — you have to call `discover_compliance_urls("Indiana")` yourself.

What if you could just *ask* it?

Below is a ~50-line conversational wrapper around the functions you've already built. It uses the OpenAI API to interpret a natural language instruction, extract intent, and call the right function. This is the minimum viable version of what OpenClaw does under the hood.

> **Notice what you have to build yourself:** the system prompt (your mini `AGENTS.md`), the intent parser, the tool routing, the response formatter. OpenClaw handles all of this for you — the gap between this cell and Part 5 is everything a production agent framework provides.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a teacher licensure compliance monitoring agent.

You have two capabilities:
1. check_compliance(states: list[str]) — check monitored URLs for changes in specific states
2. discover_urls(state: str) — search for new compliance pages for a state

When the user gives you an instruction, respond with a JSON object like:
  {"action": "check_compliance", "states": ["Ohio", "Texas"]}
  {"action": "discover_urls", "state": "Indiana"}
  {"action": "unknown", "message": "I can only check compliance or discover URLs"}

Respond with JSON only. No explanation."""

def parse_intent(user_message: str) -> dict:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

def run_agent(user_message: str):
    print(f"You: {user_message}")
    intent = parse_intent(user_message)
    action = intent.get("action")

    if action == "check_compliance":
        states = intent.get("states", [])
        print(f"Agent: Running compliance check for {', '.join(states)}...\n")
        # Import here so this cell is self-contained
        import sys; sys.path.insert(0, "custom-claw/src")
        from monitor import run_check
        changes = run_check(states)
        if changes:
            print(f"Agent: Found {len(changes)} change(s):")
            for c in changes:
                print(f"  🚨 {c['state']} — {c['subject']}: {c['url']}")
        else:
            print("Agent: No changes detected.")

    elif action == "discover_urls":
        state = intent.get("state", "")
        print(f"Agent: Searching for compliance pages for {state}...\n")
        results = discover_compliance_urls(state)
        if results:
            print(f"Agent: Found {len(results)} page(s) for {state}:")
            for r in results:
                print(f"  • {r['title']}")
                print(f"    {r['url']}")
        else:
            print("Agent: No results found (Brave API not configured).")

    else:
        print(f"Agent: {intent.get('message', 'I can check compliance or discover URLs for a state.')}")
    print()


# --- Try it ---
run_agent("Check for any licensure changes in Ohio and Texas")
run_agent("Find compliance pages for Indiana")
run_agent("What's the weather like?")

---
## Part 4: Telegram Notifications

Add `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` to your `.env` to receive alerts on your phone when compliance changes are detected.

See the [Telegram Setup](#telegram-setup) section in the README for how to create a bot and get your chat ID.

Once set, re-run the check from Part 2 — any detected change will be sent as a Telegram message.

In [ ]:
load_dotenv(override=True)

BOT_TOKEN = os.getenv("CUSTOM_CLAW_TELEGRAM_BOT_TOKEN", "")
CHAT_ID   = os.getenv("CUSTOM_CLAW_TELEGRAM_CHAT_ID", "")

if not BOT_TOKEN or not CHAT_ID:
    print("⚠️  CUSTOM_CLAW_TELEGRAM_BOT_TOKEN or CUSTOM_CLAW_TELEGRAM_CHAT_ID not set in .env")
    print("   Follow the Telegram setup in README.md, then re-run this cell.")
else:
    import requests
    resp = requests.post(
        f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
        json={"chat_id": CHAT_ID, "text": "✅ Custom Claw connected. You'll receive alerts here when licensure requirements change."},
        timeout=10
    )
    if resp.ok:
        print("✅ Telegram connected — check your phone!")
    else:
        print(f"❌ Telegram error: HTTP {resp.status_code}")